In [4]:
from reasoning_from_scratch.qwen3 import download_qwen3_small,Qwen3Tokenizer
from reasoning_from_scratch.qwen3 import Qwen3Model,QWEN_CONFIG_06_B,KVCache
from pathlib import Path
import torch.nn as nn
import torch

In [5]:
download_qwen3_small(kind="base",tokenizer_only=False,out_dir="../qwen")
##tokenizer 
tokenizer_path=Path("..")/"qwen"/"tokenizer-base.json"
tokenizer=Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)
## device
device=torch.device("cpu")
### model
model_path = Path("..") / "qwen" / "qwen3-0.6B-base.pth"
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))

✓ ..\qwen\qwen3-0.6B-base.pth already up-to-date


<All keys matched successfully>

In [16]:

def text_gen(prompt: str,model,n_tokens,eos_token_id=151643,temperature=1.0,top_k=50):
    input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0)
    outputids = torch.tensor([], dtype=torch.int)
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    with torch.no_grad():
        output = model(input_ids, cache=cache)

        for _ in range(n_tokens):

            last_logit = output[:, -1, :].squeeze(0)

            #temp scaling
            last_logit = last_logit / temperature

            # topk sampling
            k = min(top_k, last_logit.numel())
            topk_logits, topk_indices = torch.topk(last_logit, k)
            probs = torch.softmax(topk_logits, dim=-1)
            sampled_index = torch.multinomial(probs, num_samples=1)
            tokenid = topk_indices[sampled_index].reshape(1)

            if tokenid.item() == eos_token_id:
                break

            print(tokenizer.decode(tokenid.tolist()), end="", flush=True)
            yield tokenid

            input_ids = torch.concat(
                (input_ids, tokenid.unsqueeze(0)), dim=-1
            )
            output = model(tokenid.unsqueeze(0), cache=cache)



In [15]:
for token in text_gen("Write python code to generate random number",model=model, n_tokens=300):
    pass

 between 1 and 100 and ask the user to guess the number. If the user guesses the number, print "You win!" and exit the program. If the user guesses a number that is too high or too low, print "Too high" or "Too low" and ask the user to guess again. Keep track of the number of guesses the user makes and print the number of guesses made.

# Code to generate random number between 1 and 100
import random
number = random.randint(1, 100)

# Code to ask the user to guess the number
guess = int(input("Guess the number between 1 and 100: "))

# Code to keep track of the number of guesses
guesses = 1

# Code to print the number of guesses
print("Number of guesses:", guesses)

# Code to print the number of guesses
while guess != number:
    if guess > number:
        print("Too high")
    else:
        print("Too low")
    guess = int(input("Guess the number between 1 and 100: "))
    guesses += 1

# Code to print "You win!" and exit the program
if guess == number:
    print("You win!")

#### Chain of through prompting

In [13]:
for token in text_gen("Write python code to generate random number"+"Lets think step by step",model=model, n_tokens=300):
    pass


To generate a random number in Python, you can use the `random` module. Here's a step-by-step guide on how to do it:

1. Import the `random` module.
2. Use the `randint()` function to generate a random integer between two specified values. For example, to generate a random integer between 1 and 10, you would use `random.randint(1, 10)`.

Here's the Python code to generate a random number between 1 and 10:

```python
import random

# Generate a random integer between 1 and 10
random_number = random.randint(1, 10)

# Print the random number
print(random_number)
```

This code will print a random integer between 1 and 10. You can modify the range in the `randint()` function to generate random numbers between different values.